In [18]:
import os
import re
import glob
import pandas as pd

RAW_BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data"
RAW_SOI_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Raw Data/Past Holdings/Combined Past Holdings"
PROC_BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data"
PROC_SOI_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings"

os.makedirs(PROC_BASE, exist_ok=True)
os.makedirs(PROC_SOI_DIR, exist_ok=True)

UNMATCHED_LOG = os.path.join(PROC_BASE, "unmatched_names_all_files.csv")
NAME_CANDIDATES = ["security", "holding", "holding_name", "name", "company", "company_name", "holding name"]

def light_normalize(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s.strip())
    return s.upper()

def find_name_column(df: pd.DataFrame) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in NAME_CANDIDATES:
        if cand in cols_lower:
            return cols_lower[cand]
    for c in df.columns:
        if "name" in c.lower():
            return c
    raise ValueError("Could not find a company-name column")

def build_2025_map(raw_base: str) -> dict:
    mapping = {}
    paths = sorted(glob.glob(os.path.join(raw_base, "holdings_2025*.csv")))
    for p in paths:
        try:
            df = pd.read_csv(p)
        except Exception:
            continue
        try:
            name_col = find_name_column(df)
        except Exception:
            continue
        ticker_col = None
        for c in df.columns:
            if c.lower().strip() == "ticker":
                ticker_col = c
                break
        if ticker_col is None:
            continue
        df["name_normalized"] = df[name_col].map(light_normalize)
        sub = df[["name_normalized", ticker_col]].dropna(subset=["name_normalized"])
        for nm, tkr in sub.values:
            tkr = "" if pd.isna(tkr) else str(tkr).strip()
            if nm and tkr and nm not in mapping:
                mapping[nm] = tkr
    return mapping

def enrich_csv(file_path: str, out_dir: str, mapping: dict) -> tuple[str, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(file_path)
    name_col = find_name_column(df)
    df["name_normalized"] = df[name_col].map(light_normalize)
    if "company_ticker" not in df.columns:
        df["company_ticker"] = ""
    need = df["company_ticker"].eq("")
    df.loc[need, "company_ticker"] = df.loc[need, "name_normalized"].map(mapping).fillna("")
    base = os.path.splitext(os.path.basename(file_path))[0]
    out_path = os.path.join(out_dir, f"{base}_with_tickers.csv")
    df.to_csv(out_path, index=False)
    um = df.loc[df["company_ticker"].eq(""), ["name_normalized"]].drop_duplicates()
    um["source_file"] = os.path.basename(file_path)
    return out_path, df, um

def enrich_excel(file_path: str, out_dir: str, mapping: dict) -> tuple[str, pd.DataFrame, pd.DataFrame]:
    xls = pd.ExcelFile(file_path)
    sheet = xls.sheet_names[0]
    df = pd.read_excel(file_path, sheet_name=sheet)
    name_col = find_name_column(df)
    df["name_normalized"] = df[name_col].map(light_normalize)
    if "company_ticker" not in df.columns:
        df["company_ticker"] = ""
    need = df["company_ticker"].eq("")
    df.loc[need, "company_ticker"] = df.loc[need, "name_normalized"].map(mapping).fillna("")
    base = os.path.splitext(os.path.basename(file_path))[0]
    out_path = os.path.join(out_dir, f"{base}_with_tickers.xlsx")
    with pd.ExcelWriter(out_path, engine="openpyxl") as w:
        df.to_excel(w, index=False, sheet_name=sheet)
    um = df.loc[df["company_ticker"].eq(""), ["name_normalized"]].drop_duplicates()
    um["source_file"] = os.path.basename(file_path)
    return out_path, df, um

def main():
    mapping = build_2025_map(RAW_BASE)
    unmatched = []
    soi_paths = sorted(glob.glob(os.path.join(RAW_SOI_DIR, "soi_*.csv")))
    for f in soi_paths:
        try:
            out, df, um = enrich_csv(f, PROC_SOI_DIR, mapping)
            unmatched.append(um)
            print(f"[OK] Wrote: {out}")
        except Exception as e:
            print(f"[ERROR] {f}: {e}")
    holdings_2025_paths = sorted(glob.glob(os.path.join(RAW_BASE, "holdings_2025*.csv")))
    for f in holdings_2025_paths:
        try:
            out, df, um = enrich_csv(f, PROC_BASE, mapping)
            unmatched.append(um)
            print(f"[OK] Wrote: {out}")
        except Exception as e:
            print(f"[ERROR] {f}: {e}")
    cc_path = os.path.join(RAW_BASE, "Controversial_and_Clean_Holdings.xlsx")
    if os.path.exists(cc_path):
        try:
            out, df, um = enrich_excel(cc_path, PROC_BASE, mapping)
            unmatched.append(um)
            print(f"[OK] Wrote: {out}")
        except Exception as e:
            print(f"[ERROR] {cc_path}: {e}")
    if unmatched:
        all_um = pd.concat(unmatched, ignore_index=True).drop_duplicates().sort_values(["source_file", "name_normalized"])
        all_um.to_csv(UNMATCHED_LOG, index=False)
        print(f"[INFO] Unmatched log: {UNMATCHED_LOG}")
    else:
        print("[INFO] No unmatched names")

if __name__ == "__main__":
    main()


[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2017_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2018_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2019_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2020_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2021_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2022_with_tickers.csv
[OK] Wrote: /Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings/soi_2023_with_tickers.csv
[OK] Wrote: /Users/n

In [20]:
import os
import re
import glob
import json
import time
import math
import requests
import pandas as pd
from difflib import SequenceMatcher
from datetime import datetime

PROC_BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data"
PROC_SOI_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Processed Data/Combined Past Holdings"
FINAL_BASE = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data"
FINAL_SOI_DIR = "/Users/nityaarya/Downloads/Project/blackrock-esg-etf-study/Data/Final Data/Combined Past Holdings"

os.makedirs(FINAL_BASE, exist_ok=True)
os.makedirs(FINAL_SOI_DIR, exist_ok=True)

YAHOO_SEARCH_URL = "https://query2.finance.yahoo.com/v1/finance/search"
YAHOO_SLEEP_SECONDS = 0.4
YAHOO_TIMEOUT = 8.0
MIN_SIMILARITY = 0.60
PREFERRED_EXCHANGES = {"NASDAQ", "NYSE", "NYSE ARCA", "NYSE MKT", "NASDAQGS", "NASDAQCM", "NASDAQGM"}

NAME_CANDIDATES = ["security", "holding", "holding_name", "name", "company", "company_name", "holding name"]

def light_normalize(s: str) -> str:
    if pd.isna(s):
        return ""
    s = str(s).replace("\u00A0", " ")
    s = re.sub(r"\s+", " ", s.strip())
    return s.upper()

def find_name_column(df: pd.DataFrame) -> str:
    cols_lower = {c.lower(): c for c in df.columns}
    for cand in NAME_CANDIDATES:
        if cand in cols_lower:
            return cols_lower[cand]
    for c in df.columns:
        if "name" in c.lower():
            return c
    raise ValueError("Could not find a company-name column")

def similarity(a: str, b: str) -> float:
    return SequenceMatcher(None, a, b).ratio()

def choose_best_quote(query_original: str, quotes: list) -> str:
    q_norm = light_normalize(query_original)
    best = None
    best_score = -1.0
    for q in quotes:
        symbol = (q.get("symbol") or "").strip()
        if not symbol or len(symbol) > 12:
            continue
        qt = (q.get("quoteType") or "").upper()
        exch = (q.get("exchDisp") or q.get("exchange") or "").upper()
        yname = (q.get("shortname") or q.get("longname") or "").strip()
        y_norm = light_normalize(yname)
        sim = similarity(q_norm, y_norm)
        score = sim * 10.0
        if qt == "EQUITY":
            score += 2.5
        if exch in PREFERRED_EXCHANGES:
            score += 1.5
        if q_norm and y_norm and (q_norm in y_norm or y_norm in q_norm):
            score += 1.0
        if score > best_score:
            best_score = score
            best = (symbol, sim)
    if best is None:
        return ""
    symbol, sim = best
    return symbol if sim >= MIN_SIMILARITY else ""

def yahoo_lookup(session: requests.Session, company_name: str) -> str:
    try:
        params = {"q": company_name, "quotesCount": 10, "newsCount": 0, "lang": "en-US", "region": "US"}
        r = session.get(YAHOO_SEARCH_URL, params=params, timeout=YAHOO_TIMEOUT)
        if r.status_code != 200:
            return ""
        data = r.json()
        quotes = data.get("quotes", []) or []
        if not quotes:
            return ""
        return choose_best_quote(company_name, quotes)
    except Exception:
        return ""

def ensure_name_norm_and_ticker(df: pd.DataFrame) -> pd.DataFrame:
    name_col = find_name_column(df)
    if "name_normalized" not in df.columns:
        df["name_normalized"] = df[name_col].map(light_normalize)
    else:
        df["name_normalized"] = df["name_normalized"].map(light_normalize)
    if "company_ticker" not in df.columns:
        df["company_ticker"] = ""
    df["company_ticker"] = df["company_ticker"].fillna("").astype(str)
    return df

def fill_with_yahoo(df: pd.DataFrame, session: requests.Session, progress_cb=None, file_tag="") -> pd.DataFrame:
    name_col = find_name_column(df)
    mask = df["company_ticker"].eq("")
    if not mask.any():
        return df
    queries = df.loc[mask, name_col].fillna("").astype(str).str.strip()
    uniq = queries.drop_duplicates()
    total = len(uniq)
    done = 0
    results = {}
    start = time.time()
    for val in uniq:
        sym = yahoo_lookup(session, val)
        results[val] = sym
        done += 1
        if progress_cb:
            elapsed = max(time.time() - start, 1e-6)
            rate = done / elapsed
            remain = (total - done) / rate if rate > 0 else 0
            progress_cb(file_tag, "yahoo", done, total, elapsed, remain)
        time.sleep(YAHOO_SLEEP_SECONDS)
    df.loc[mask, "company_ticker"] = df.loc[mask, name_col].map(results).fillna("").replace("nan", "")
    return df

def synth_base_from_name(nm: str) -> str:
    keep = "".join(ch for ch in nm if ch.isalpha())
    if not keep:
        keep = "".join(ch for ch in nm if ch.isalnum())
    if not keep:
        keep = "TICK"
    base = keep[:5]
    if len(base) < 3:
        base = (base + "X"*3)[:3]
    return base

def build_synthetic_map(dfs: list) -> dict:
    all_blank_names = []
    for df in dfs:
        if "name_normalized" in df.columns and "company_ticker" in df.columns:
            sub = df.loc[df["company_ticker"].eq(""), "name_normalized"].dropna().astype(str).tolist()
            all_blank_names.extend(sub)
    all_blank_names = pd.unique(pd.Series(all_blank_names))
    mapping = {}
    used = set()
    for nm in all_blank_names:
        base = synth_base_from_name(nm)
        cand = base
        k = 1
        while cand in used:
            k += 1
            cand = f"{base}{k}"
        mapping[nm] = cand
        used.add(cand)
    return mapping

def apply_synthetic_map(df: pd.DataFrame, synth_map: dict) -> pd.DataFrame:
    mask = df["company_ticker"].eq("")
    if mask.any():
        df.loc[mask, "company_ticker"] = df.loc[mask, "name_normalized"].map(synth_map).fillna("")
    return df

def save_soi_final(path_in: str, df: pd.DataFrame):
    base = os.path.basename(path_in)
    m = re.search(r"(soi_(\d{4}))", base, re.IGNORECASE)
    if m:
        outname = f"{m.group(1).lower()}_final.csv"
    else:
        stem = os.path.splitext(base)[0]
        outname = f"{stem.replace('_with_tickers','')}_final.csv"
    out_path = os.path.join(FINAL_SOI_DIR, outname)
    df.to_csv(out_path, index=False)
    print(f"[FINAL] {out_path}", flush=True)

class Tracker:
    def __init__(self, total_tasks: int):
        self.total = total_tasks
        self.done = 0
        self.t0 = time.time()
    def tick(self, msg: str):
        self.done += 1
        pct = (self.done / self.total) * 100 if self.total else 100.0
        elapsed = time.time() - self.t0
        rate = self.done / elapsed if elapsed > 0 else 0
        remain = (self.total - self.done) / rate if rate > 0 else 0
        bar_len = 24
        filled = int(bar_len * self.done / max(self.total, 1))
        bar = "█" * filled + "░" * (bar_len - filled)
        print(f"[{bar}] {self.done}/{self.total} {pct:5.1f}% | {msg} | elapsed {elapsed:6.1f}s | eta {remain:6.1f}s", flush=True)

def per_file_progress(file_tag, phase, done, total, elapsed, remain):
    pct = (done / total) * 100 if total else 100.0
    bar_len = 20
    filled = int(bar_len * (done / max(total, 1)))
    bar = "#" * filled + "-" * (bar_len - filled)
    print(f"  [{bar}] {phase}: {done}/{total} ({pct:4.1f}%) | {file_tag} | elapsed {elapsed:5.1f}s | eta {remain:5.1f}s", flush=True)

def main():
    print(f"[START] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", flush=True)
    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; ESGTickerBot/1.0)", "Accept": "application/json,text/javascript,*/*;q=0.1", "Connection": "keep-alive"})

    soi_paths = sorted(glob.glob(os.path.join(PROC_SOI_DIR, "soi_*_with_tickers.csv")))
    if not soi_paths:
        soi_paths = sorted(glob.glob(os.path.join(PROC_SOI_DIR, "soi_*.csv")))
    holdings_paths = sorted(glob.glob(os.path.join(PROC_BASE, "holdings_2025*_with_tickers.csv")))
    if not holdings_paths:
        holdings_paths = sorted(glob.glob(os.path.join(PROC_BASE, "holdings_2025*.csv")))
    cc_path = None
    for p in ["Controversial_and_Clean_Holdings_with_tickers.xlsx", "Controversial_and_Clean_Holdings.xlsx"]:
        cand = os.path.join(PROC_BASE, p)
        if os.path.exists(cand):
            cc_path = cand
            break

    total_tasks = len(soi_paths) + len(holdings_paths) + (1 if cc_path else 0) + len(soi_paths) + len(holdings_paths) + (1 if cc_path else 0)
    tracker = Tracker(total_tasks if total_tasks > 0 else 1)

    soi_dfs, soi_meta = [], []
    for p in soi_paths:
        try:
            df = pd.read_csv(p)
            df = ensure_name_norm_and_ticker(df)
            tracker.tick(f"loaded {os.path.basename(p)}")
            before_blanks = int(df["company_ticker"].eq("").sum())
            df = fill_with_yahoo(df, session, per_file_progress, os.path.basename(p))
            after_blanks = int(df["company_ticker"].eq("").sum())
            tracker.tick(f"yahoo filled {os.path.basename(p)} (blanks {before_blanks}→{after_blanks})")
            soi_dfs.append(df)
            soi_meta.append(p)
        except Exception as e:
            tracker.tick(f"error {os.path.basename(p)}: {e}")

    holdings_dfs, holdings_meta = [], []
    for p in holdings_paths:
        try:
            df = pd.read_csv(p)
            df = ensure_name_norm_and_ticker(df)
            tracker.tick(f"loaded {os.path.basename(p)}")
            before_blanks = int(df["company_ticker"].eq("").sum())
            df = fill_with_yahoo(df, session, per_file_progress, os.path.basename(p))
            after_blanks = int(df["company_ticker"].eq("").sum())
            tracker.tick(f"yahoo filled {os.path.basename(p)} (blanks {before_blanks}→{after_blanks})")
            holdings_dfs.append(df)
            holdings_meta.append(p)
        except Exception as e:
            tracker.tick(f"error {os.path.basename(p)}: {e}")

    cc_df = None
    cc_tag = None
    if cc_path:
        try:
            xls = pd.ExcelFile(cc_path)
            sheet = xls.sheet_names[0]
            cc_df = pd.read_excel(cc_path, sheet_name=sheet)
            cc_df = ensure_name_norm_and_ticker(cc_df)
            tracker.tick(f"loaded {os.path.basename(cc_path)}")
            before_blanks = int(cc_df["company_ticker"].eq("").sum())
            cc_df = fill_with_yahoo(cc_df, session, per_file_progress, os.path.basename(cc_path))
            after_blanks = int(cc_df["company_ticker"].eq("").sum())
            tracker.tick(f"yahoo filled {os.path.basename(cc_path)} (blanks {before_blanks}→{after_blanks})")
            cc_tag = os.path.basename(cc_path)
        except Exception as e:
            tracker.tick(f"error {os.path.basename(cc_path)}: {e}")

    synth_map = build_synthetic_map(soi_dfs + holdings_dfs + ([cc_df] if cc_df is not None else []))

    for p, df in zip(soi_meta, soi_dfs):
        try:
            pre = int(df["company_ticker"].eq("").sum())
            df = apply_synthetic_map(df, synth_map)
            post = int(df["company_ticker"].eq("").sum())
            save_soi_final(p, df)
            tracker.tick(f"synthetic & saved {os.path.basename(p)} (blanks {pre}→{post})")
        except Exception as e:
            tracker.tick(f"error saving {os.path.basename(p)}: {e}")

    for p, df in zip(holdings_meta, holdings_dfs):
        try:
            pre = int(df["company_ticker"].eq("").sum())
            df = apply_synthetic_map(df, synth_map)
            post = int(df["company_ticker"].eq("").sum())
            out_path = os.path.join(FINAL_BASE, "holdings_2025_final.csv" if re.search(r"holdings_2025", os.path.basename(p), re.IGNORECASE) else os.path.basename(p).replace("_with_tickers", "_final"))
            df.to_csv(out_path, index=False)
            print(f"[FINAL] {out_path}", flush=True)
            tracker.tick(f"synthetic & saved {os.path.basename(out_path)} (blanks {pre}→{post})")
        except Exception as e:
            tracker.tick(f"error saving {os.path.basename(p)}: {e}")

    if cc_df is not None:
        try:
            pre = int(cc_df["company_ticker"].eq("").sum())
            cc_df = apply_synthetic_map(cc_df, synth_map)
            post = int(cc_df["company_ticker"].eq("").sum())
            out_xlsx = os.path.join(FINAL_BASE, "Controversial_and_Clean_Holdings_final.xlsx")
            with pd.ExcelWriter(out_xlsx, engine="openpyxl") as w:
                cc_df.to_excel(w, index=False, sheet_name="Sheet1")
            print(f"[FINAL] {out_xlsx}", flush=True)
            tracker.tick(f"synthetic & saved {os.path.basename(out_xlsx)} (blanks {pre}→{post})")
        except Exception as e:
            tracker.tick(f"error saving {os.path.basename(cc_path)}: {e}")

    print(f"[DONE] {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", flush=True)

if __name__ == "__main__":
    main()


[START] 2025-10-15 20:14:06
[█░░░░░░░░░░░░░░░░░░░░░░░] 1/20   5.0% | loaded soi_2017_with_tickers.csv | elapsed    0.0s | eta    0.8s
  [--------------------] yahoo: 1/1749 ( 0.1%) | soi_2017_with_tickers.csv | elapsed   1.3s | eta 2224.7s
  [--------------------] yahoo: 2/1749 ( 0.1%) | soi_2017_with_tickers.csv | elapsed   2.1s | eta 1833.4s
  [--------------------] yahoo: 3/1749 ( 0.2%) | soi_2017_with_tickers.csv | elapsed   2.8s | eta 1625.3s
  [--------------------] yahoo: 4/1749 ( 0.2%) | soi_2017_with_tickers.csv | elapsed   3.8s | eta 1636.5s
  [--------------------] yahoo: 5/1749 ( 0.3%) | soi_2017_with_tickers.csv | elapsed   4.4s | eta 1551.6s
  [--------------------] yahoo: 6/1749 ( 0.3%) | soi_2017_with_tickers.csv | elapsed   5.1s | eta 1472.5s
  [--------------------] yahoo: 7/1749 ( 0.4%) | soi_2017_with_tickers.csv | elapsed   5.8s | eta 1442.1s
  [--------------------] yahoo: 8/1749 ( 0.5%) | soi_2017_with_tickers.csv | elapsed   6.6s | eta 1429.0s
  [---------------